# Data and features

* 입력: 구매 로그·고객·상품·캠페인 테이블
* 피처: 기존 35개 + 기준일 이전 포인트 피처 3개
* 확인 자료: 공개 코드·입력 점검 출력
* 원본 키 확인: CSV가 있는 경우에만 선택 실행

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

candidates = [p for base in (Path.cwd(), *Path.cwd().parents) for p in (base, base / "code")]
CODE = next(p for p in candidates if (p / "evidence").is_dir() and (p / "modeling").is_dir())
PROJECT = CODE.parent

import ast

def literal_assignment(path, name):
    tree = ast.parse(path.read_text(encoding="utf-8"))
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == name for t in node.targets):
            return ast.literal_eval(node.value)
    raise KeyError(name)

base_columns = literal_assignment(CODE / "modeling/build_features.py", "COLUMNS")
redeem_columns = literal_assignment(CODE / "modeling/add_redeem_b.py", "NEW_COLUMNS")
assert len(base_columns) - 2 == 35  # client_id와 treatment_flg 제외
assert len(redeem_columns) == 3
summary = pd.DataFrame({"구성": ["구매·고객·상품 피처", "기준일 이전 포인트 피처", "모델 입력 합계"],
                        "개수": [35, 3, 38]})
display(summary)
print("추가 피처:", ", ".join(redeem_columns))

,구성,개수
0,구매·고객·상품 피처,35
1,기준일 이전 포인트 피처,3
2,모델 입력 합계,38


추가 피처: redeem_by_cutoff, days_redeem_to_cutoff, issue_redeem_delay_cens


## 데이터 통합·전처리 실행 (선택)

* 순서: 구매 로그 집계 → 포인트 피처 생성 → train/holdout·CV fold 분할
* 입력: `X5_RAW_DATA_DIR`의 원본 CSV 4개
* 출력: `X5_WORK_DIR`의 비공개 작업 폴더; 공개 코드 폴더에는 저장하지 않음
* 실행 조건: 두 경로 지정 후 `X5_RUN_FULL=1`


In [ ]:
import hashlib
import os
import shutil
import subprocess
import sys

def file_sha(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def prepare_work_dir(raw_dir, work_dir):
    assert Path(work_dir).is_absolute(), "X5_WORK_DIR: 절대 경로 필요"
    raw_dir, work_dir = Path(raw_dir).resolve(), Path(work_dir).resolve()
    assert work_dir != raw_dir and work_dir not in raw_dir.parents
    assert all((raw_dir / name).is_file() for name in
               ("clients.csv", "products.csv", "purchases.csv", "uplift_train.csv"))
    assert work_dir.is_absolute() and work_dir != PROJECT and PROJECT not in work_dir.parents
    assert work_dir != CODE and CODE not in work_dir.parents
    for source in (CODE / "modeling").rglob("*.py"):
        target = work_dir / "modeling" / source.relative_to(CODE / "modeling")
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            assert file_sha(target) == file_sha(source), f"코드 불일치: {target}"
        else:
            shutil.copy2(source, target)
    source = CODE / "env/requirements.lock.txt"
    target = work_dir / "env/requirements.lock.txt"
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert file_sha(target) == file_sha(source), f"환경 잠금 파일 불일치: {target}"
    else:
        shutil.copy2(source, target)
    link = work_dir / "data/og/retailhero-uplift/data"
    link.parent.mkdir(parents=True, exist_ok=True)
    if link.exists() or link.is_symlink():
        assert link.resolve() == raw_dir, f"원본 경로 불일치: {link}"
    else:
        link.symlink_to(raw_dir, target_is_directory=True)
    return work_dir

def run_step(work_dir, name, *args):
    log = work_dir / "logs" / f"{name}.log"
    log.parent.mkdir(parents=True, exist_ok=True)
    with log.open("w") as output:
        subprocess.run([sys.executable, str(work_dir / "modeling" / f"{name}.py"), *args],
                       cwd=work_dir, stdout=output, stderr=subprocess.STDOUT, check=True)
    print(name, "완료 ·", log)

if os.environ.get("X5_RUN_FULL") == "1":
    assert os.environ.get("X5_RAW_DATA_DIR") and os.environ.get("X5_WORK_DIR")
    WORK = prepare_work_dir(os.environ["X5_RAW_DATA_DIR"], os.environ["X5_WORK_DIR"])
    run_step(WORK, "build_features", "--cutoff", "2019-03-18 23:40:03",
             "--out", "data/processed/features_cut.parquet")
    run_step(WORK, "add_redeem_b")
    run_step(WORK, "split")
    run_step(WORK, "split_final_folds")
    for name in ("features_full38.parquet",):
        table = pd.read_parquet(WORK / "data/processed" / name)
        print(name, table.shape, "· target 제외:", "target" not in table.columns)
    split = pd.read_parquet(WORK / "data/splits/split.parquet")
    display(split["split"].value_counts().rename("고객 수").to_frame())
else:
    print("전체 전처리 실행: 생략 · 저장된 점검 결과 확인")


## 입력 시점 점검

* 포인트 최초 사용일: 구매 관측 종료 후 갱신된 기록 포함
* 처리: 기준일 이전 사용 여부·경과일로 재구성
* 아래 출력: 저장된 시점·처치 균형 점검 기록; 원시 로그 재집계 결과 아님

In [2]:
timing = (CODE / "evidence/checks/redeem_timing.out.txt").read_text()
balance = (CODE / "evidence/checks/redeem_balance.out.txt").read_text()
for line in timing.splitlines():
    if line.startswith(("purchases rows", "window ", "train N", "redeem > window end", "redeem missing")):
        print(line)
print()
for line in balance.splitlines():
    if any(label in line for label in ("base35+redeem3", "base35+raw3", "VERDICT:")):
        print(line)

purchases rows 45786568
window 2018-11-21 21:02:33 2019-03-18 23:40:03
train N 200039 treat share 0.49981
redeem > window end                      n=  22600 (11.30%) treat=0.5084 z=+2.59
redeem missing                           n=  17546 ( 8.77%) treat=0.4809 z=-5.01
redeem missing & spent points within window: 0 of 17546

  base35+redeem3                   p=40 AUC=0.4998  z_vs_0.5=-0.14  (6.3s)
  base35+raw3 (positive control)   p=40 AUC=0.5189  z_vs_0.5=+14.66  (6.3s)
  |z| base35+redeem3 < 3.0     ok
VERDICT: PASS


## 원본 입력 확인 (선택)

* 입력: `X5_RAW_DATA_DIR`에 지정한 원시 CSV 폴더
* 확인: 고객·상품·캠페인 테이블의 키 결측·중복
* 제외: 구매 로그 4,578만 행 재집계, 고객 식별자·개별 구매 결과 출력

In [3]:
import os
raw = Path(os.environ.get("X5_RAW_DATA_DIR", str(CODE / "data/og/retailhero-uplift/data")))
required = {"clients.csv": "client_id", "products.csv": "product_id", "uplift_train.csv": "client_id"}
if not all((raw / name).is_file() for name in [*required, "purchases.csv"]):
    print("원시 CSV: 미지정 · 입력 키 점검: 생략")
else:
    rows = []
    for name, key in required.items():
        keys = pd.read_csv(raw / name, usecols=[key], dtype={key: "string"})[key]
        rows.append({"테이블": name, "행 수": len(keys), "키 결측": int(keys.isna().sum()),
                     "중복 키": int(keys.duplicated().sum())})
    display(pd.DataFrame(rows))
    print("구매 로그 전체 읽기: 생략")

원시 CSV: 미지정 · 입력 키 점검: 생략
